# 03 — Per-segment analyzer fields & retry behavior

The classifier+analyzer call from notebook 02 is non-deterministic for
sparse tables: every so often a segment returns zero rows even though the
PDF clearly has them. `sec_service.classify_and_extract()` retries the
whole call (up to `MAX_RETRIES=2`) when any non-Other segment came back
empty.

This notebook shows the retry probe and inspects the analyzer's row-level
output (`level`, `isSectionHeader`, `isSubtotal`, `values[]`).


In [ ]:
from _lib import sec_service
import time

sample = sec_service.list_samples()[0]['file_name']
sec_service.ensure_analyzers()

t0 = time.time()
pdf_bytes = (sec_service.ATTACH_DIR / sample).read_bytes()
raw, retries = sec_service.classify_and_extract(pdf_bytes, max_retries=2)
print(f'wall: {time.time()-t0:.1f}s   retries: {retries}')
empties = sec_service._has_empty_tables(raw)
print('still-empty segments after retries:', empties)

## First few rows from each non-Other segment

In [ ]:
for seg in raw.get('contents', []):
    cat = seg.get('category', 'Other')
    if cat == 'Other':
        continue
    ft = (seg.get('fields') or {}).get('financialTables', {}) or {}
    for j, tbl_raw in enumerate(ft.get('valueArray', []) or [], 1):
        tobj = tbl_raw.get('valueObject', {}) or {}
        title = (tobj.get('tableTitle') or {}).get('valueString', '')
        rows = (tobj.get('rows') or {}).get('valueArray', []) or []
        print(f"\n--- {cat} table {j}: {title[:60]} ({len(rows)} rows)")
        for r in rows[:5]:
            ro = r.get('valueObject', {}) or {}
            line = (ro.get('lineItem') or {}).get('valueString', '')
            level = (ro.get('level') or {}).get('valueString', '')
            hdr = (ro.get('isSectionHeader') or {}).get('valueString', '')
            sub = (ro.get('isSubtotal') or {}).get('valueString', '')
            print(f"   L{level} hdr={hdr} sub={sub}  {line[:80]}")